<font size="+3">Autoencoders <small>&</small> Variational Autoencoders</font>

---

This tutorial has two parts: the first is dedicated to autoencoders and the second to their variational counterparts.

The goal is to demonstrate how to manipulate autoencoders and variational autoencoders using a simple example: the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database. The goal is to visualize the concepts discussed in class, particularly the notion of latent space.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random as rd
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset, ConcatDataset
from torchvision import datasets, transforms
from torchsummary import summary

In [ ]:
from tqdm import tqdm
#from tqdm.notebook import tqdm

In [ ]:
from sklearn.manifold import TSNE
from scipy.stats import norm 

This code lines allow you to check if your computer is using CPU or GPU ressources.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

---
# The [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database

As seen in previous tutorials, algorithm stability is enhanced by data normalization. In pytorch, you can apply transformations to the data when you load it: hence what we do here.

In [ ]:
# Transform: convert images to Tensor
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert pixel values to [0,1] + explicit channel dimension
])

# Load datasets
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [ ]:
print(f"Train set: {len(train_dataset)} images of size {train_dataset.data.shape[1]} x {train_dataset.data.shape[2]}")

print(f"Test set:  {len(test_dataset)} images")

In order to be able to train neural networks more easily afterwards, we create a `DataLoader` to access the data. In particular, we need to specify the size of (future) training batches.

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size = BATCH_SIZE, 
    shuffle = True, 
    num_workers = 2,
    pin_memory = True)

test_loader = DataLoader(
    test_dataset, 
    batch_size = BATCH_SIZE, 
    shuffle = False,
    num_workers = 2,
    pin_memory = True)

You can then access a batch with the command `next(iter(train_loader))`.

In [ ]:
images, labels = next(iter(train_loader))

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

The following code displays sample images.

In [ ]:
images, labels = next(iter(train_loader))  # One batch
images = images.to("cpu")  # ensure on CPU
n = 10

plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i+1)
    plt.imshow(images[i][0], cmap="gray")
    ax.grid(False)
    plt.axis("off")
plt.show()

---
# Autoencoders

## A first very simple autoencoder

First, we will build a very simple architecture where :

* The **encoder**: is a dense layer of 32 neurons (the latent variable dimension) with an activation function $\texttt{reLu}$:
$$\texttt{reLu}(x) = max(0,x) \,,$$

* The **decoder**: is a dense layer of $784=28\times28$ neurons (the input dimension) with a sigmoid activation function
$$\sigma(x) = \frac{1}{1+\text{e}^x} \,.$$.

##### <i style="color:purple">**Exercise**: write the simple model described above</i>

In [ ]:
### TO BE COMPLETED ###

n_input = 784
n_latent = 32

In [ ]:
### TO BE COMPLETED ###

class SimpleAutoencoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super(SimpleAutoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(n_input, n_latent),
            nn.ReLU()
        )
        # Decoder
        self.decoder = ... ### TO BE COMPLETED
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Instantiate model
autoencoder = SimpleAutoencoder(n_input, n_latent)
summary(autoencoder, input_size=(n_input,))

In [ ]:
# %load solutions/ae/SimpleAutoencoder.py

We can now train the model. Note that _the target variable is the original image._

In [ ]:
n_input = 784
n_latent = 32

EPOCHS = 10  #50
LEARNING_RATE = 1e-3

In [ ]:
# Device configuration
simple_autoencoder = SimpleAutoencoder(n_input, n_latent)
simple_autoencoder.to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(simple_autoencoder.parameters(), lr=LEARNING_RATE)

# --- #
# Training loop
for epoch in range(EPOCHS):
    simple_autoencoder.train()
    train_loss = 0
    
    for inputs, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        # Move data to the same device as model
        inputs = inputs.to(device)
        inputs_flat = inputs.view(inputs.size(0), -1)
        
        optimizer.zero_grad()
        outputs = simple_autoencoder(inputs_flat)
        loss = criterion(outputs, inputs_flat)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    simple_autoencoder.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            inputs_flat = inputs.view(inputs.size(0), -1)
            outputs = simple_autoencoder(inputs_flat)
            loss = criterion(outputs, inputs_flat)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(test_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

**Note**: Here we have used binary cross-entropy as the loss function, following the seminal article [[Kingma & Welling, 2014]](https://arxiv.org/pdf/1312.6114.pdf). This choice is legitimate here because of the _sigmoid_ activation function.

##### <i style="color:purple">**Exercise**: Check the model's performance by viewing sample images from the test database and their reconstruction.</i>

To do this, write a `plot_images` function that displays the images contained in the $\texttt{imgs}$ list.
* $\texttt{imgs}$ contains a list, each of its items is itself a list of images;
* Each of the images in these sub-lists must be displayed on the same row of the global image;
* In particular, at the end, the global image contains as many columns as the size of each of the $\texttt{imgs}$ items (they all have the same size, i.e. $n$), and as many rows as the size of $\texttt{imgs}$.

> This function is used in the global `visualize_autoencoder` function defined immediately afterwards, which uses a series of auxiliary functions to :
> * Check that the autoencoder being "visualized" is on the right device, thanks to `check_device_model`;
> * Randomly select $n$ images from the test dataset using `select_n_samples`;
> * display these images, their encoded and decoded counterparts using the `plot_images` function, which you will or have just coded.

You can freely use the `check_device_model` and `select_n_samples` functions later.

In [ ]:
### TO BE COMPLETED ###

def plot_images(imgs, sz, titles, n, cmap="gray"):
    num_rows = len(imgs)
    plt.figure(figsize=(2*n, 2*num_rows))

    for row, images in enumerate(imgs):
        # Move to CPU and convert to numpy for plotting
        images = images.cpu().numpy()
        
        [...]
            
        plt.subplot(num_rows, n, row*n+1).set_title(titles[row], fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
# %load solutions/ae/plot_images.py

In [ ]:
def check_device_model(model, device):
    # Move autoencoder to device only if needed
    current_device = next(model.parameters()).device
    if current_device != torch.device(device):
        model.to(device)
    return model
    

# --- #

def select_n_samples(laoder, n, device):
    dataset = laoder.dataset
    indices = torch.randperm(len(dataset))[:n]
    inputs_list = [dataset[i][0].unsqueeze(0) for i in indices]
    inputs = torch.cat(inputs_list, dim=0).to(device)
    return inputs
    

# --- #

def visualize_autoencoder(autoencoder, 
                          loader = test_loader, 
                          input_sz = (28,28), 
                          latent_sz = (4,8),
                          n = 10, 
                          device = device
                         ):

    autoencoder = check_device_model(autoencoder, device=device)
    autoencoder.eval()
    
    inputs = select_n_samples(laoder=loader, n=n, device=device)  # [n, 1, 28, 28]
    inputs_flat = inputs.view(inputs.size(0), -1)                 # flatten for encoder/decoder

    with torch.no_grad():
        encoded = autoencoder.encoder(inputs_flat)
        decoded = autoencoder.decoder(encoded)

    plot_images(
        imgs = [inputs, encoded, decoded],
        sz = [input_sz, latent_sz, input_sz],
        titles = ['Original', 'Encoded', 'Decoded'],
        n = n
    )

In [ ]:
visualize_autoencoder(simple_autoencoder)

##### <i style="color:purple">**Exercise**: Check that you get the same results from decoding latent representations as from encoding-decoding original data.</i>

Write a function to display, on three different lines and for a dozen images
- On the 1st line, the original image;
- On the 2nd line, the “auto-encoded” image, i.e. its global transformation after passing through the auto-encoder itself;
- On the 3rd line, the image encoded by the encoder, then decoded by the decoder.
- 
To do this, use the `visualize_autoencoder` function defined above (especially for synthaxe).

In [ ]:
### TO BE COMPLETED ###

def check_encode_decode(autoencoder, 
                        test_loader = test_loader, 
                        input_sz = (28,28), 
                        n = 10,
                        device = device
                       ):

    [...]

In [ ]:
# %load solutions/ae/check_encode_decode.py

In [ ]:
check_encode_decode(simple_autoencoder)

## A more realistic autoencoder

We decided to make the architecture _slightly_ more complex

In [ ]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        self.fc1 = nn.Linear(n_input, 128)
        self.fc2 = nn.Linear(128, n_latent) 

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Decoder
class Decoder(nn.Module):
    def __init__(self, n_latent, n_output):
        super().__init__()
        self.fc1 = nn.Linear(n_latent, 128)
        self.fc2 = nn.Linear(128, n_output)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        self.encoder = Encoder(n_input, n_latent)
        self.decoder = Decoder(n_latent, n_input)

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


# Instantiate models
encoder = Encoder(n_input, n_latent)
decoder = Decoder(n_latent, n_input)
autoencoder = Autoencoder(n_input, n_latent)

# Print summaries
print("Autoencoder")
summary(autoencoder, (n_input,))
print("\n Encoder")
summary(encoder, (n_input,))
print("\n Decoder")
summary(decoder, (n_latent,))

We perform the training with a loop similar to the one we defined for `simple_autoencoder`.

In [ ]:
EPOCHS = 10  #50
LEARNING_RATE = 1e-3

# Device configuration
autoencoder = Autoencoder(n_input, n_latent)
autoencoder.to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)

# --- #
# Training loop
for epoch in range(EPOCHS):
    autoencoder.train()
    train_loss = 0
    
    for inputs, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        inputs = inputs.to(device)
        inputs_flat = inputs.view(inputs.size(0), -1)
        
        optimizer.zero_grad()
        outputs = autoencoder(inputs_flat)
        loss = criterion(outputs, inputs_flat)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    autoencoder.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            inputs_flat = inputs.view(inputs.size(0), -1)
            outputs = autoencoder(inputs_flat)
            loss = criterion(outputs, inputs_flat)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(test_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# --- #
visualize_autoencoder(autoencoder)

While we have increased the complexity of the model, the results are less convincing.

##### <i style="color:purple">**Exercise**:  Why do you think this is? Using the first model as a guide, suggest an improvement that would produce better results.</i>

Take advantage of this opportunity to define a `train_autoencoder` function to train an auto-encoder, so you can easily train other auto-encoders in the future.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/Autoencoder_improved.py

In [ ]:
# Decoder
class Decoder_improved(nn.Module):
    def __init__(self, n_latent, n_output):
        super().__init__()
        self.fc1 = nn.Linear(n_latent, 128)
        self.fc2 = nn.Linear(128, n_output)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# Autoencoder
class Autoencoder_improved(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        self.encoder = Encoder(n_input, n_latent)
        self.decoder = Decoder_improved(n_latent, n_input)

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
def train_autoencoder(autoencoder,
                      train_loader = train_loader,
                      test_loader = test_loader,
                      criterion = nn.BCELoss(),
                      EPOCHS = 10,  #50
                      LEARNING_RATE = 1e-3,
                      device = device,
                      use_tqdm = True
                     ):

    autoencoder = check_device_model(autoencoder, device=device)
    optimizer = optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)
    
    # --- #
    # Training loop
    for epoch in range(EPOCHS):
        autoencoder.train()
        train_loss = 0

        iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False) if use_tqdm else train_loader
        for inputs, _ in iterator:
            inputs = inputs.to(device)
            inputs_flat = inputs.view(inputs.size(0), -1)
            
            optimizer.zero_grad()
            outputs = autoencoder(inputs_flat)
            loss = criterion(outputs, inputs_flat)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
        train_loss /= len(train_loader.dataset)
        
        # Validation
        autoencoder.eval()
        val_loss = 0
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.to(device)
                inputs_flat = inputs.view(inputs.size(0), -1)
                outputs = autoencoder(inputs_flat)
                loss = criterion(outputs, inputs_flat)
                val_loss += loss.item() * inputs.size(0)
        val_loss /= len(test_loader.dataset)
        
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
    return autoencoder, train_loss, val_loss

In [ ]:
autoencoder = Autoencoder_improved(n_input, n_latent)
autoencoder, _, _ = train_autoencoder(autoencoder)
visualize_autoencoder(autoencoder)

### Influence of latent space dimension

With the latent space dimension provided, a (relatively) low reconstruction error is observed. We'd like to study the influence of the latent space dimension on the reconstruction error.

##### <i style="color:purple">**Exercise**: Draw a _curve_ (with only a few points) showing the evolution of the reconstruction error as a function of the latent space dimension.</i>

What is the minimum dimension of the latent space that still allows us to observe a reasonable reconstruction of the data (with the network provided)?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/latent_dimension.py

In the previous example, the autoencoder is constrained only by the size of the hidden layer. We could have envisaged other ways of constraining this space, for example by introducing parsimony constraints (sparse auto-encoders). We do not address this issue here.

## Convolutional autoencoders

In the previous sections, we saw very simple autoencoders where the encoder and decoder parts are perceptrons. As seen in the course, they can both be composed of more layers and different types of layers.

In particular, convolutional layers are the best layers to use when dealing with images.

##### <i style="color:purple">**Exercise**: Implement a convolutional autoencoder with the following architecture</i>

**Encoder:**
* Two convolution layers, 16 filters of size 3x3
* A maxpooling layer with 2x2 filters
* Two convolution layers, 8 filters of size 3x3
* A maxpooling layer with 2x2 filters

**Decoder:**
* Two convolution layers, 8 filters of size 3x3
* An upsampling layer with 2x2 filters
* Two convolution layers, 16 filters of size 3x3
* An upsampling layer with 2x2 filters
* A convolution layer, 1 filter of size 3x3, with $\texttt{sigmoid}$ activation.

All padding is $\texttt{same}$ and all convolution layer activation functions, except the last, are $\texttt{reLu}$.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/ConvAutoencoder.py

##### <i style="color:purple">**Exercise**: Write the counterpart of the `train_autoencoder` function adapted to the convolutional autoencoder defined above.</i>

Pay particular attention to the size of the network inputs.

In [ ]:
### TO BE COMPLETED ###

def train_convolutional_autoencoder(autoencoder,
                                    train_loader = train_loader,
                                    test_loader = test_loader,
                                    criterion = nn.BCELoss(),
                                    EPOCHS = 10,  #50
                                    LEARNING_RATE = 1e-3,
                                    device = device,
                                    use_tqdm = True
                                   ):

    [...]

In [ ]:
# %load solutions/ae/train_convolutional_autoencoder.py

In [ ]:
### TO BE COMPLETED ###

def visualize_convolutional_autoencoder(autoencoder, 
                          loader = test_loader, 
                          input_sz = (28,28), 
                          latent_sz=(14,28),
                          n = 10, 
                          device = device
                         ):

    [...]

In [ ]:
# %load solutions/ae/visualize_convolutional_autoencoder.py

In [ ]:
convolutional_autoencoder = ConvAutoencoder()
convolutional_autoencoder, _, _ = train_convolutional_autoencoder(convolutional_autoencoder, use_tqdm=True)
visualize_convolutional_autoencoder(convolutional_autoencoder)

## Application to denoising

We now know how to build a convolutional auto-encoder. Let's now see how to use it to solve an image denoising problem.
First, we create false noisy data from MNIST data.

In [ ]:
class NoisyMNIST(Dataset):
    def __init__(self, original_dataset, noise_factor=0.4):
        self.dataset = original_dataset
        self.noise_factor = noise_factor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        noisy_img = img + self.noise_factor * torch.randn_like(img)
        noisy_img = torch.clamp(noisy_img, 0., 1.)
        return noisy_img, img, label

In [ ]:
train_noisy_dataset = NoisyMNIST(train_dataset)
test_noisy_dataset = NoisyMNIST(test_dataset)

# --- #

train_noisy_loader = DataLoader(
    train_noisy_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 0,
    pin_memory = True
)

test_noisy_loader = DataLoader(
    test_noisy_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 0,
    pin_memory = True
)

Let's take a look at the noise we created.

##### <i style="color:purple">**Exercise**: Display a few images and their noisy counterparts.</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/noisy_images.py

##### <i style="color:purple">**Exercise**: Adapt the `train_convolutional_autoencoder` function to the dataset induced by the `NoisyMNIST` class.</i>

In [ ]:
### TO BE COMPLETED ###

def train_noisy_convolutional_autoencoder(autoencoder,
                                          train_loader = train_noisy_loader,
                                          test_loader = test_noisy_loader,
                                          criterion = nn.BCELoss(),
                                          EPOCHS = 10,  #50
                                          LEARNING_RATE = 1e-3,
                                          device = device,
                                          use_tqdm = True
                                         ):

    [...]

In [ ]:
# %load solutions/ae/train_noisy_convolutional_autoencoder.py

In [ ]:
convolutional_autoencoder = ConvAutoencoder()
convolutional_autoencoder, _, _ = train_noisy_convolutional_autoencoder(convolutional_autoencoder, use_tqdm=True)
visualize_convolutional_autoencoder(convolutional_autoencoder, loader=test_noisy_loader)

## Semi-supervised learning: Classification <small style="color:orangered">(to go further)</small>

The idea of semi-supervised learning in this context is to take advantage of the latent space's ability to represent data well (or so we hope).

Specifically, we have a small labeled dataset, and a large unlabeled dataset. We will compare two classifiers: one trained on the original data and another trained on the encoded data. Thus, if the latent space is ideally constructed, the second classifier should outperform the first (unfortunately, this is not the case, given the simplicity of the networks we have defined so far, and the ease of the task).

The first step is to create small subsets of labeled data. We keep $n=100$ data per digit.

In [ ]:
convolutional_autoencoder = ConvAutoencoder()
convolutional_autoencoder, _, _ = train_convolutional_autoencoder(convolutional_autoencoder, use_tqdm=True)

In [ ]:
nb_train = 100
nb_test = 50

train_indices = []
test_indices  = []

for i in range(10):
    class_indices = [idx for idx, (_, label) in enumerate(train_dataset) if label == i]    
    train_indices.extend(class_indices[:nb_train])
    test_indices.extend(class_indices[-nb_test:])

train_indices = torch.tensor(train_indices)[torch.randperm(len(train_indices))]
test_indices  = torch.tensor(test_indices)[torch.randperm(len(test_indices))]

few_train_dataset = Subset(train_dataset, train_indices)
few_test_dataset  = Subset(train_dataset, test_indices)

We then encode this subset of data. To do this, we use the convolutional autoencoder trained on all the data.

In [ ]:
def encode_dataset(autoencoder, dataset, batch_size=64):
    # Move autoencoder to device only if needed
    current_device = next(autoencoder.parameters()).device
    if current_device != torch.device(device):
        autoencoder.to(device)
    autoencoder.eval()
    
    encoded_list, labels_list = [], []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            encoded = autoencoder.encoder(inputs)
            encoded_list.append(encoded.cpu())
            labels_list.append(labels)
    
    return torch.cat(encoded_list), torch.cat(labels_list)

few_encoded_train, few_labels_train = encode_dataset(convolutional_autoencoder, few_train_dataset)
few_encoded_test,  few_labels_test  = encode_dataset(convolutional_autoencoder, few_test_dataset)

We can then define, and train, a classifier on the latent space.

In [ ]:
class LatentClassifier(nn.Module):
    def __init__(self, latent_dim, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(latent_dim, n_classes)
    def forward(self, x):
        return self.fc(x)


def train_classifier(classifier, dataset, EPOCHS=30, BATCH_SIZE=20):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    classifier.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = classifier(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.shape[0]
        total_loss /= len(loader.dataset)
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")
    return classifier


few_encoded_train_flat = few_encoded_train.view(few_encoded_train.size(0), -1)
few_encoded_test_flat  = few_encoded_test.view(few_encoded_test.size(0), -1)
latent_dim = few_encoded_train_flat.shape[1]

latent_classifier = LatentClassifier(latent_dim=latent_dim).to(device)
latent_dataset    = TensorDataset(few_encoded_train_flat, few_labels_train)
latent_classifier = train_classifier(latent_classifier, latent_dataset)

In [ ]:
def evaluate_classifier(classifier, dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = classifier(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += outputs.shape[0]
    acc = correct / total
    return acc


latent_acc = evaluate_classifier(latent_classifier, TensorDataset(few_encoded_test_flat, few_labels_test))
print(f"Accuracy on latent test set: {latent_acc:.4f}")

Finally, we define and train a classifier on the original space.

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, input_dim, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_classes)
    def forward(self, x):
        return self.fc(x)

def flatten_dataset(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    for inputs, labels in loader:
        inputs_flat = inputs.view(inputs.shape[0], -1)
        return inputs_flat, labels

inputs_few_train, labels_few_train = flatten_dataset(few_train_dataset)
inputs_few_test,  labels_few_test  = flatten_dataset(few_test_dataset)

image_classifier = ImageClassifier(inputs_few_train.shape[1]).to(device)
image_dataset    = TensorDataset(inputs_few_train, labels_few_train)
image_classifier = train_classifier(image_classifier, image_dataset)

image_acc = evaluate_classifier(image_classifier, TensorDataset(inputs_few_test, labels_few_test))
print(f"\n Accuracy on image test set: {image_acc:.4f}")

---
# Variational Autoencoders

In this section, we will build a simple variational auto-encoder and apply it to (i) number generation and (ii) anomaly detection. We rely on the same dataset as in the first part, _i.e._ [MNIST](https://en.wikipedia.org/wiki/MNIST_database).

<br>
<div><img src="img/vae_mnist.png" width="600px" style="display:block; margin-left:auto; margin-right:auto;"/></div>

## Building the variational autoencoder

### Encoder

First we build the **encoder**. It consists of :
1. A dense layer of `intermediate_dim = 512` neurons, with activation function $\texttt{ReLu}$ ;
2. Two dense layers of `latent_dim = 2` neurons **above the same 1st layer**, with a linear activation function. These two layers will produce the two variables `z_mean` and `z_log_var` in latent space.

In [ ]:
# network parameters
input_dim = 28 * 28
intermediate_dim = 512
latent_dim = 2

##### <i style="color:purple">**Exercise**: Write a `PyTorch` code implementing the encoder described above.</i>

In [ ]:
### TO BE COMPLETED ###

class Encoder(nn.Module):
    
    [...]

In [ ]:
# %load solutions/vae/Encoder.py

#### Stochastic latent variable

We'll use the reparametrization trick to define the latent random variable $z$ conditional on the input image $x$ according to the normal distribution:
$$ z\vert x \sim \mathcal{N}\big(\mu_z(x), \sigma_z(x)\big) \,. $$

<br>
<div><img src="img/vae_3.svg" width="600px" style="display:block; margin-left:auto; margin-right:auto;"/></div>
<br>

The reparametrization trick is to redefine $z$ as follows:
$$ z\vert x=\mu_z(x)+\sigma_z(x)\cdot\varepsilon\qquad\text{with}\qquad\varepsilon\sim\mathcal{N}(0,1) \,. $$

In this way, the dependency between $z$ and $x$ becomes deterministic and _differentiable_. Moreover, the randomness of $z$, at a fixed $x$, is carried solely by $\varepsilon$.

This leads us to modify the Encoder class as follows:

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, intermediate_dim, latent_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, intermediate_dim)
        self.fc_mean = nn.Linear(intermediate_dim, latent_dim)
        self.fc_log_var = nn.Linear(intermediate_dim, latent_dim)

    def reparameterize(self, mean, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mean + eps * std

    def forward(self, x):
        h = F.relu(self.fc1(x))
        z_mean = self.fc_mean(h)
        z_log_var = self.fc_log_var(h)
        z = self.reparameterize(z_mean, z_log_var)
        return z, z_mean, z_log_var


encoder = Encoder(input_dim, intermediate_dim, latent_dim)
summary(encoder, (input_dim,))

**Note** : Using the exponential function in the `reparameterize` function ensures that the standard deviation $\texttt{std}$ is positive.

### Decoder

The decoder takes the vector $z$ (_i.e._, the sample from the latent distribution defined by the encoder) as input. It then consists of two dense layers with the following characteristics:
* `intermediate_dim = 512` neurons, $\texttt{ReLu}$ activation.
* `input_dim = 784` neurons, sigmoid activation.

##### <i style="color:purple">**Exercise**: Build this decoder.</i>

In [ ]:
### TO BE COMPLETED ###

class Decoder(nn.Module):
    
    [...]

In [ ]:
# %load solutions/vae/Decoder.py

### Autoencoder

We can now combine the encoder and decoder to define our variational autoencoder.

##### <i style="color:purple">**Exercise**: Build this VAE.</i>

In [ ]:
### TO BE COMPLETED ###

class VAE_mlp(nn.Module):
    
    [...]

In [ ]:
# %load solutions/vae/VAE_mlp.py

#### Loss function

We now implement the VAE loss function as described in the course:
$$ \mathcal{L}_{VAE} \,=\, \mathcal{L}(x,\hat{x}) \,+\, KL\big(\, q(z\vert x) \,\vert\vert\, p(z) \,\big) \,, $$
where $\mathcal{L}$ is a loss function adapted to our problem between the original image $x$ and the reconstructed image $\hat{x}$. As seen in the first part of this lab, we choose binary cross-entropy. 
$KL$ denotes the Kullback-Leibler divergence, and $p$ denotes the prior distribution that we place on the latent variable $z$. 
Given that
$$ z\vert x\sim\mathcal{N}(\mu_z(x),\sigma_z(x)), $$
we choose as the prior for $z$ the reduced centered Gaussian distribution: $z\sim\mathcal{N}(0,1)\,$.

<p style="color:teal">
<b>Proposition</b>: Consider two Gaussians $\, \mathcal{N}_d(\mu_p,\Sigma_p) \,$ and $\, \mathcal{N}_d(\mu_q,\Sigma_q)$. Then their K-L divergence is given by:
$$
KL\big(\, \mathcal{N}_d(\mu_q,\Sigma_q) \,\vert\vert\, \mathcal{N}_d(\mu_p,\Sigma_p) \big) \,=\, \frac12 \left[
\log\frac{\vert\Sigma_p\vert}{\vert\Sigma_q\vert} - d + \textrm{tr}\big(\Sigma_p^{-1}\Sigma_q \big) + \big(\mu_p -
\mu_q\big)^\top \Sigma_p^{-1}\big(\mu_p - \mu_q\big)\right] \,.
$$
</p>

<br>

In our context, $\, z\sim\mathcal{N}_2(0,I_2) \,$ et $\, z\vert x\sim\mathcal{N}_2(\mu_z(x),\sigma_z(x)) \,$. Thus, we get:
$$ 
KL\big(\, q(z\vert x) \,\vert\vert\, p(z) \,\big) \,=\, \frac12 \left[-\log\vert\sigma_z(x)\vert - 2 + \text{tr} \big(\sigma_z(x)\big) +\mu_z(x)^\top \mu_z(x)\right] \,,
$$
or even
$$
KL\big(\, q(z\vert x) \,\vert\vert\, p(z) \,\big) \,=\, \frac12 \, \sum_{j=1}^2 \Big[ \sigma_{z,j}(x) + \mu_{z,j}^2(x) - 1 - \log\sigma_{z,j}(x) \Big] \,.
$$

##### <i style="color:purple">**Exercise**: Using this last expression, define the loss function of the VAE.</i>

In [ ]:
### TO BE COMPLETED ###

class vae_loss(x, x_hat, mean, log_var):
    
    [...]

In [ ]:
# %load solutions/vae/vae_loss.py

### Training and Results


In [ ]:
input_dim = 28 * 28
intermediate_dim = 512
latent_dim = 2

LEARNING_RATE = 1e-3
EPOCHS = 20

##### <i style="color:purple">**Exercise**: Using the codes provided in the first part as inspiration, train the VAE.</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/vae_mlp_train.py

We can now quickly verify the model's performance by viewing examples of images from the test database and their reconstructions.

In [ ]:
def reconstruct_batch(model, loader, device=device):
    model.eval()
    x, _ = next(iter(loader))
    x = x.view(x.size(0), -1).to(device)   # flatten
    with torch.no_grad():
        x_hat, _, _ = model(x)
    x_hat = x_hat.view(-1, 1, 28, 28)
    x = x.view(-1, 1, 28, 28)
    return x, x_hat

# --- #
x, x_hat = reconstruct_batch(vae, test_loader)
plot_images(
        imgs = [x, x_hat],
        sz = [(28, 28), (28, 28)],
        titles = ['Original', 'Decoded'],
        n = 10
    )

## Training visualization

In order to visualize the training, we would like to display the evolution of the figures during the training.

In [ ]:
n = 10  # number of images to visualize
m = 10  # number of epochs to display

##### <i style="color:purple">**Exercise**: Complete the following code to display the first $m$ training steps for $n$ images.</i>

* Each step, including initialization and real image, will be represented on a single row;
* Each column corresponds to an image, transformed step by step;
* In particular, the complete figure has $n$ columns and $m+2$ rows.

In [ ]:
### TO BE COMPLETED ###

# Init model and optimizer
vae_seq = VAE_mlp(input_dim, intermediate_dim, latent_dim).to(device)
optimizer = torch.optim.Adam(..., lr=LEARNING_RATE)  ### TO BE COMPLETED

# --- #  
# Select test images for plotting
idx = torch.randint(0, len(test_dataset), (n,))

test_images = []
for i in idx:
    img, _ = test_dataset[i]
    test_images.append(img)
x_test_sample = torch.stack(test_images).view(n, -1).to(device)

imgs = [x_test_sample]
titles = ['Images']


# --- #  
# Reconstruction before training (epoch 0)
vae_seq.eval()
with torch.no_grad():
    x_test_decoded = vae_seq(x_test_sample)[0].view(n, 28, 28)

imgs.append(x_test_decoded)
titles.append('Init')


# --- #
# Training loop
for j in range(m):
    print(f"=== Epoch {j+1}/{m} ===", end="\r", flush=True)
    vae_seq.train()
    for x_batch, _ in train_loader:
        x_batch = x_batch.view(x_batch.size(0), -1).to(device)
        optimizer.zero_grad()
        x_hat, z_mean, z_log_var = ...  ### TO BE COMPLETED
        loss = vae_loss(x_batch, x_hat, z_mean, z_log_var)
        loss.backward()
        optimizer.step()

    # Reconstruction at epoch j
    vae_seq.eval()
    with torch.no_grad():
        x_test_decoded = vae_seq(x_test_sample)[0].view(n, 28, 28)

    imgs.append(...)   ### TO BE COMPLETED
    titles.append(...) ### TO BE COMPLETED
    
print("Training complete!")

sz = [(28,28)] * len(imgs)
plot_images(imgs, sz, titles, n)

In [ ]:
# %load solutions/vae/training_visualization.py

## Classification of the latent variable

What is the distribution of the different numbers on the latent space?

In order to study the compromise induced by the reconstruction/kl balance in the loss, we introduce the following loss. We refer to $\beta$-VAE when training a VAE with this loss.

In [ ]:
def beta_vae_loss(x, x_hat, mean, log_var, beta=1):
    reconstruct_loss = F.binary_cross_entropy(x_hat, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruct_loss + beta * kl

##### <i style="color:purple">**Exercise**: Represent the scatter plot of the encoded digits (i.e., their latent representation), with each point colored according to the value of its label.</i>

We can consider several values of $\beta$.

To help you, start by writing a `train_beta_VAE` function that trains a $\beta$-VAE for a given $\beta$.

In [ ]:
### TO BE COMPLETED ###

def train_beta_vae(vae,
              beta = 1,
              train_loader = train_loader,
              test_loader = test_loader,
              EPOCHS = 10,  #30
              LEARNING_RATE = 1e-3,
              device = device,
              use_tqdm = True
             ):
    
    vae = check_device_model(vae, device=device)
    optimizer = optim.Adam(vae.parameters(), lr=LEARNING_RATE)

    [...]

    return vae, train_loss, val_loss

In [ ]:
# %load solutions/vae/train_beta_vae.py

In [ ]:
# %load solutions/vae/plot_latents_1.py

In [ ]:
# %load solutions/vae/plot_latents_2.py

## New number generation

The generative nature of the VAE can be used to generate new data, in this case new numbers.

##### <i style="color:purple">**Exercise**: Generate a new image.</i>

To do this, we can generate a realization of the random latent variable $z$ and use the decoder.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/new_number.py

You can also visualize the manifold related to the latent space.

In [ ]:
n = 15 # figure with 15x15 panels
digit_size = 28

figure = np.zeros((digit_size * n, digit_size * n))
grid_x = norm.ppf(np.linspace(0.05, 0.95, n))
grid_y = norm.ppf(np.linspace(0.05, 0.95, n))

decoder.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_x):
        for j, xi in enumerate(grid_y):
            z_sample = np.array([[xi, yi]], dtype=np.float32)
            z_sample = torch.from_numpy(z_sample)
            x_decoded = vae.decoder(z_sample)
            digit = x_decoded[0].cpu().numpy().reshape(digit_size, digit_size)

            figure[
                i * digit_size: (i+1) * digit_size,
                j * digit_size: (j+1) * digit_size
            ] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="gray")
plt.axis("off")
plt.show()

## Anomaly Detection

In this section, we will see how variational autoencoders can be used for anomaly detection.
To do this, we will consider the number 9 to be an outlier. We will generate four datasets from the training and test datasets, namely:
* Training and test data without the 9,
* Training and test data with only images of 9.

In [ ]:
BATCH_SIZE = 64
outlier_i = 9

In [ ]:
def make_dataset(x, y, label, target_class=outlier_i, batch_size=BATCH_SIZE):
    mask = (y == target_class) if label == 1 else (y != target_class)
    x_select = x[mask]
    n = x_select.shape[0] - x_select.shape[0] % batch_size  # ensure full batches
    x_select = x_select[:n]
    x_select = x_select.unsqueeze(1).float() / 255.0  # counterparts of transforms.ToTensor() function (cf cell 1)
    y_select = torch.full((n,), label, dtype=torch.long)
    dataset = TensorDataset(x_select, y_select)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=(label==0))
    return dataset, loader

# --- Raw data ---
x_train, y_train = train_dataset.data, train_dataset.targets
x_test, y_test = test_dataset.data, test_dataset.targets

# --- DataLoaders for Anomaly Detection ---
_, train_ad_loader              = make_dataset(x_train, y_train, label=0)  # normal train
test_ad_dataset, test_ad_loader = make_dataset(x_test, y_test, label=0)    # normal test
_, train_anomaly_loader                    = make_dataset(x_train, y_train, label=1)  # anomaly train
test_anomaly_dataset, test_anomaly_loader  = make_dataset(x_test, y_test, label=1)    # anomaly test

test_ad_anomaly_dataset = ConcatDataset([test_ad_dataset, test_anomaly_dataset])
test_ad_anomaly_loader = DataLoader(test_ad_anomaly_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
EPOCHS = 30
vae = VAE_mlp(input_dim, intermediate_dim, latent_dim).to(device)
vae, _, _ = train_beta_vae(vae, beta=1,
                           train_loader=train_ad_loader, test_loader=test_ad_loader,
                           EPOCHS=EPOCHS, use_tqdm=False)

### Images decoded by the VAE
Let's now use our VAE on the test dataset with known numbers (0 to 8) on the one hand, and on outliers (9) on the other.

##### <i style="color:purple">**Exercise**: Compare the reconstruction of regular and outliers data for the VAE trained only on regular data.</i>

* View examples of images from the regular test database and their reconstructions,
* Do the same for outliers.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/reconstructions_comparison.py

### Anomaly detection using latent representation

By performing “Normal-vs-Outliers” clustering in the latent space, we can hope to detect anomalies.

##### <i style="color:purple">**Exercise**: Compare the distribution of encodings for regular and outlier data.</i>

* To do this, we can visualize the scatter plot corresponding to normal data on one side and outlier data on the other, on the same graph.
* Can we easily implement a clustering technique?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/ad_detection_latents.py

### Distribution of reconstruction error

To evaluate the performance of image detection $9$, let's check the distribution of the various $\ell_2$ errors between the original image and its reconstructed image for three types of images:
* Images with a known number ($0$ to $8$),
* Aberrant images ($9$),
* Images generated completely at random.

##### <i style="color:purple">**Exercise**: Construct a histogram of reconstruction errors for each of the three situations.</i>

In [ ]:
### TO BE COMPLETED ###

print("=== MSE over differents dataset ===")

# --- #
# Regular data
[...]
mse_regular = ...

# --- #
# Outliers data
[...]
mse_outliers = ...

# --- #
# Random data
[...]
mse_random = ...

In [ ]:
# %load solutions/vae/MSE_anomaly.py

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/histograms_anomaly.py

##### <i style="color:purple">**Question**: What can you say about error reconstruction in different cases?</i>

Conclude on the performance of the VAE in detecting anomalies.

The performance of such a classifier can also be assessed using a ROC curve.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
mse_score = np.concatenate([mse_regular, mse_outliers], 0)
true_label = [0]*regular_data.shape[0] + [1]*outliers_data.shape[0]

if roc_auc_score(true_label, mse_score)<0.5 :
    mse_score *= -1
    
fpr, tpr, thresholds = roc_curve(true_label, mse_score)
auc_score = roc_auc_score(true_label, mse_score)

fig, ax = plt.subplots(1, 1, figsize = (9,5))

ax.plot(fpr, tpr, 'c.-', label = 'ROC Curve {:2.2f}'.format(auc_score))
ax.plot(fpr, fpr, 'k-', label = 'Random Guessing')

ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
plt.legend()
plt.show()

Clearly, the result is not very conclusive here. We need to improve the architecture of our VAE to increase performance!

## Convolutional Variational Autoencoders

We have seen how to build a VAE and how to use it to generate new images and detect anomalies. The VAEs used so far only use dense layers (MLP), which may explain their lack of effectiveness in anomaly detection. 

##### <i style="color:purple">**Question**: Use CNN layers to build a convolutional VAE and test the different applications (generating images and detecting anomalies).</i>